# Experiment 8a: coincidence / resonance transport (bypass and repair)

See `experiments/experiment_8a_design.md`.

Not another denominator. Two levers on the **transport** wall:
- **bypass**: an additive, transport-independent similarity term `lambda * sim(v|q)`;
- **repair**: relation-type routing (`RelatedTo` as a weak conductor) + a genericness penalty.

Plus the `7c` overlap method as a **baseline control** (expected to tie), and
**manifold-aware** evaluation (score a target *manifold*, not one canonical node).

Controlled source (manual anchors), sources excluded from ranking, ConceptNet-only
(ATOMIC off).


In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
SRC = Path.cwd() / "src"
if not (SRC / "memory").exists():
    SRC = Path.cwd().parent  # allow running from src/experiments
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print("src on path:", SRC)

src on path: c:\Users\yaros\BlackSabbitch_repo\RIN\src


In [2]:
import time
from collections import defaultdict
import numpy as np

from embryonic import conceptnet_source
from embryonic.loader import EmbryonicGraphLoader
from embryonic.numberbatch import Numberbatch
from embryonic.query import query_vectors  # reused for sanity if needed

## Config

Seed terms are the union of every cue / target / distractor below, so the seeded
subgraph actually contains them. Radius small; ATOMIC off.

In [3]:
SOURCE_DIR = Path.cwd() / "source"
if not SOURCE_DIR.exists():
    SOURCE_DIR = Path.cwd().parent.parent / "source"
CONCEPTNET_DUMP = SOURCE_DIR / "conceptnet-assertions-5.7.0.csv.gz"
NUMBERBATCH_FILE = SOURCE_DIR / "numberbatch-en-19.08.txt.gz"
LANGUAGE = "en"

SEED_TERMS = [
    "coffee", "stimulant", "caffeine", "beverage", "hot beverage", "energy", "tea", "espresso", "drink coffee",
    "check in", "document", "passport", "boarding pass", "airport", "suitcase", "identity",
    "laptop", "battery", "electricity", "charger", "power", "power adapter", "office", "work",
    "wake up", "tired", "morning", "sleep",
    "apartment", "hotel", "neighbor", "rent", "home",
    "trip", "travel", "vacation", "pack", "flight",
]
RADIUS = 2
LIMIT_PER_NODE = 40
MAX_NODES = 12000

# scoring hyperparameters (single starting point; sweep lambda below)
ETA = 0.25         # PageRank prior strength
MU = 1.0           # genericness penalty strength
C_RELATEDTO = 0.2  # RelatedTo conductance (repair): << 1.0
LAMBDAS = [0.0, 0.5, 1.0, 2.0, 4.0]  # additive-sim sweep (0.0 == transport-only)
LAMBDA_MAIN = 1.0
DECAY = 0.8
MIN_PROP_FRAC = 0.01
MAX_STEPS = 20

## Build once: ConceptNet adjacency index (~150s) and Numberbatch (~70s)

In [4]:
t0 = time.time()
adjacency = conceptnet_source.build_language_adjacency(str(CONCEPTNET_DUMP), LANGUAGE)
print(f"adjacency: {len(adjacency):,} concepts in {time.time()-t0:.0f}s")

adjacency: 1,787,373 concepts in 153s


In [5]:
t0 = time.time()
numberbatch = Numberbatch()
numberbatch.load(str(NUMBERBATCH_FILE))
nterms = len(getattr(numberbatch, "terms", getattr(numberbatch, "vectors", {})))
print(f"numberbatch: {nterms:,} terms in {time.time()-t0:.0f}s")

numberbatch: 516,782 terms in 76s


## Build the seeded ConceptNet-only graph and attach embeddings

In [6]:
loader = EmbryonicGraphLoader(
    entries=SEED_TERMS, radius=RADIUS, language=LANGUAGE,
    conceptnet_backend="local_dump", conceptnet_adjacency=adjacency,
    limit_per_node=LIMIT_PER_NODE, max_nodes=MAX_NODES,
    include_atomic=False, numberbatch=numberbatch,
)
t0 = time.time()
G = loader.load()
print(f"graph: {len(G.nodes)} nodes in {time.time()-t0:.1f}s")

# ensure every concept node has an embedding (fallback: look up its URI)
filled = 0
for node in G.nodes.values():
    if node.embedding is None and node.provenance:
        uri = node.provenance[0]
        if isinstance(uri, str) and uri.startswith("/c/"):
            v = numberbatch.vector_for(uri)
            if v is not None:
                node.embedding = v
                filled += 1
embedded = sum(1 for n in G.nodes.values() if n.embedding is not None)
print(f"embeddings present: {embedded}/{len(G.nodes)} (filled {filled} via URI)")

graph: 12000 nodes in 0.2s
embeddings present: 6821/12000 (filled 0 via URI)


## Indices + anchor resolution

In [7]:
uri2id, content2id = {}, {}
for nid, node in G.nodes.items():
    if node.provenance and isinstance(node.provenance[0], str):
        uri2id.setdefault(node.provenance[0], nid)
    if node.content:
        content2id.setdefault(node.content.strip().lower(), nid)

def norm(term):
    return term.strip().lower().replace(" ", "_")

def resolve(term):
    """term -> node id, trying /c/en/<norm> then bare content."""
    uri = f"/c/{LANGUAGE}/{norm(term)}"
    if uri in uri2id:
        return uri2id[uri]
    return content2id.get(term.strip().lower())

def resolve_group(terms):
    """first resolvable term in a cue-group -> (node_id, term)."""
    for t in terms:
        nid = resolve(t)
        if nid is not None:
            return nid, t
    return None, None

NODE_IDS = list(G.node_ids())
print("sample resolves:", {t: resolve(t) for t in ["coffee", "caffeine", "beverage", "charger", "passport"]})

sample resolves: {'coffee': 0, 'caffeine': 2, 'beverage': 18, 'charger': 486, 'passport': 250}


## Pre-step (free): anisotropy diagnostic

If mean pairwise cosine of random node pairs is ~0.3-0.5, the space is a narrow
cone (weak margins) and 8b is worth doing. If ~0, cosine is fine.

In [8]:
rng = np.random.default_rng(0)
emb_ids = [n for n in NODE_IDS if G.nodes[n].embedding is not None]
M = np.array([np.asarray(G.nodes[n].embedding, dtype=np.float32) for n in emb_ids])
Mn = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-12)
pairs = rng.integers(0, len(emb_ids), size=(20000, 2))
pairs = pairs[pairs[:, 0] != pairs[:, 1]]
cos = np.sum(Mn[pairs[:, 0]] * Mn[pairs[:, 1]], axis=1)
print(f"mean pairwise cosine = {cos.mean():.3f}  (std {cos.std():.3f})")
print("-> narrow cone / anisotropic; 8b (de-coning) worthwhile" if cos.mean() > 0.15 else "-> roughly isotropic; cosine is fine")

mean pairwise cosine = 0.038  (std 0.093)
-> roughly isotropic; cosine is fine


## Query suite (controlled source, cue-groups, target manifold, named distractor)

Each cue-group resolves to one anchor. `manifold` = acceptable graph-native
targets; `canonical` = first of them; `distractor` = the generic node that must
lose on primary queries (or win on controls).

In [9]:
QUERIES = [
  # kind, id, cue_groups, manifold (acceptable targets), distractors (all must lose)
  ("primary", "Q1", [["coffee"], ["stimulant", "energy"]],
      ["caffeine", "stimulant"], ["beverage", "hot beverage", "popular drink"]),
  ("primary", "Q2", [["check in", "checking in"], ["document"]],
      ["passport", "boarding pass"], ["airport", "checking in", "storing information"]),
  ("primary", "Q3", [["laptop"], ["battery", "electricity"]],
      ["charger", "power adapter"], ["office", "power"]),           # bare \'power\' now a distractor to BEAT, not a soft target
  ("primary", "Q7", [["wake up"], ["tired", "sleepy"]],
      ["coffee", "espresso", "drink coffee"], ["morning", "bed"]),  # distractor no longer a source anchor
  ("control", "Q5", [["apartment", "neighbor"], ["rent", "home"]],
      ["apartment", "home"], ["trip"]),
  ("control", "Q6", [["trip", "travel"], ["vacation", "flight"]],
      ["trip", "travel"], ["apartment"]),
]

for kind, qid, groups, manifold, distractors in QUERIES:
    res = [resolve_group(g) for g in groups]
    anchors = [r[1] for r in res]
    anchor_ids = {r[0] for r in res if r[0] is not None}
    man_ok = [t for t in manifold if resolve(t) is not None and resolve(t) not in anchor_ids]
    dist_ok = [t for t in distractors if resolve(t) is not None and resolve(t) not in anchor_ids]
    dist_bad = [t for t in distractors if resolve(t) is not None and resolve(t) in anchor_ids]
    warn = f"  !! distractor is a source (dropped): {dist_bad}" if dist_bad else ""
    print(f"{qid} [{kind}] anchors={anchors} manifold_ok={man_ok} distractors_ok={dist_ok}{warn}")

Q1 [primary] anchors=['coffee', 'stimulant'] manifold_ok=['caffeine'] distractors_ok=['beverage', 'hot beverage', 'popular drink']
Q2 [primary] anchors=['check in', 'document'] manifold_ok=['passport', 'boarding pass'] distractors_ok=['airport', 'checking in', 'storing information']
Q3 [primary] anchors=['laptop', 'battery'] manifold_ok=['charger', 'power adapter'] distractors_ok=['office', 'power']
Q7 [primary] anchors=['wake up', 'tired'] manifold_ok=['coffee', 'espresso', 'drink coffee'] distractors_ok=['morning', 'bed']
Q5 [control] anchors=['apartment', 'rent'] manifold_ok=['home'] distractors_ok=['trip']
Q6 [control] anchors=['trip', 'vacation'] manifold_ok=['travel'] distractors_ok=['apartment']


## Operators

Local diffusion mirrors `src/memory/diffuse.py` plus two hooks: a per-relation
`conductance` (repair) and a per-destination `rel` mask (the gate). Kept in the
notebook so `src` stays untouched until a method is promoted.

In [10]:
RELATEDTO = {"relatedto", "related_to"}

def diffuse_local(G, sources, decay=DECAY, min_prop_frac=MIN_PROP_FRAC,
                  max_steps=MAX_STEPS, restart=0.0, rel=None, c_relatedto=1.0):
    current = dict(sources)
    peak = dict(current)
    floor = min_prop_frac * (max(sources.values()) if sources else 0.0)
    for _ in range(max_steps):
        active = {n: a for n, a in current.items() if a >= floor}
        if not active:
            break
        nxt = defaultdict(float)
        for n, a in active.items():
            oe = G.out_edges(n)
            ws = []
            for e in oe:
                w = e.weight * (c_relatedto if (e.type or "").lower() in RELATEDTO else 1.0)
                if rel is not None:
                    w *= rel.get(e.dst, 1.0)
                ws.append(w)
            tot = sum(ws)
            if tot <= 0:
                continue
            for e, w in zip(oe, ws):
                nxt[e.dst] += decay * a * (w / tot)
        if restart > 0:
            for s, amt in sources.items():
                nxt[s] += restart * amt
        current = dict(nxt)
        for n, a in current.items():
            if a > peak.get(n, 0.0):
                peak[n] = a
    return peak

def direct_sim(query_vecs):
    """node_id -> max cosine of node.embedding to any query vector, clipped to [0,1]."""
    if not query_vecs:
        return {n: 0.0 for n in NODE_IDS}
    Q = np.array(query_vecs, dtype=np.float32)
    Q = Q / (np.linalg.norm(Q, axis=1, keepdims=True) + 1e-12)
    sims = {n: 0.0 for n in NODE_IDS}
    S = Mn @ Q.T  # (n_embedded, g)
    best = S.max(axis=1)
    for i, nid in enumerate(emb_ids):
        sims[nid] = float(max(0.0, best[i]))
    return sims

In [11]:
print("computing PageRank prior (scipy-free power iteration)...")
t0 = time.time()

def compute_pagerank(G, ids, alpha=0.85, iters=100, tol=1e-9):
    idx = {n: i for i, n in enumerate(ids)}
    N = len(ids)
    out = [G.out_edges(n) for n in ids]
    wsum = [sum(max(e.weight, 1e-12) for e in oe) for oe in out]
    pr = np.full(N, 1.0 / N)
    tele = (1.0 - alpha) / N
    for _ in range(iters):
        new = np.full(N, tele)
        dangling = 0.0
        for i, oe in enumerate(out):
            if wsum[i] <= 0:
                dangling += pr[i]
                continue
            c = alpha * pr[i] / wsum[i]
            for e in oe:
                j = idx.get(e.dst)
                if j is not None:
                    new[j] += c * max(e.weight, 1e-12)
        new += alpha * dangling / N
        if np.abs(new - pr).sum() < tol:
            pr = new
            break
        pr = new
    return {ids[i]: float(pr[i]) for i in range(N)}

PR = compute_pagerank(G, NODE_IDS)
PR = {n: max(PR.get(n, 1e-12), 1e-12) for n in NODE_IDS}
print(f"pagerank done in {time.time()-t0:.1f}s")

# genericness: short content, RelatedTo-heavy neighborhood, stoplist, high PR percentile
STOP = {"thing", "place", "person", "object", "make", "good", "one", "back",
        "home", "every", "stuff", "item", "entity", "part", "way"}
pr_arr = np.array([PR[n] for n in NODE_IDS])
pr_pct = dict(zip(NODE_IDS, pr_arr.argsort().argsort() / max(len(pr_arr) - 1, 1)))

def genericness(nid):
    c = (G.nodes[nid].content or "")
    oe = G.out_edges(nid)
    rel_frac = (sum(1 for e in oe if (e.type or "").lower() in RELATEDTO) / len(oe)) if oe else 0.0
    short = 1.0 if (len(c.split()) <= 1 and len(c) <= 4) else 0.0
    stop = 1.0 if c.lower() in STOP else 0.0
    return 1.0 + 0.5 * rel_frac + 0.5 * short + 2.0 * stop + 1.0 * pr_pct[nid]

GEN = {n: genericness(n) for n in NODE_IDS}

computing PageRank prior (scipy-free power iteration)...
pagerank done in 1.9s


## Scoring methods

- `M0`  pure baseline: `transport / prior^eta`, NO similarity (so M0->M1 isolates the additive term).
- `M_gate`  legacy control: gated diffusion (rel-mask in-propagation) = current pipeline.
- `M1`  bypass: `(transport_plain + lambda*sim) / PR^eta`.
- `M2`  bypass + repair: transport with `RelatedTo` downweighted.
- `M3`  + genericness penalty.
- `CTRL_overlap` (7c F3): per-cue geometric overlap / PR^eta  (expected to tie).

In [12]:
def query_setup(groups):
    resolved = [resolve_group(g) for g in groups]
    anchor_ids = [r[0] for r in resolved if r[0] is not None]
    src = {a: 100.0 for a in anchor_ids}
    qvecs = [np.asarray(G.nodes[a].embedding, dtype=np.float32)
             for a in anchor_ids if G.nodes[a].embedding is not None]
    return anchor_ids, src, qvecs

def pr_pow(nid, eta=ETA):
    return max(PR[nid], 1e-12) ** eta

def scores_for_query(groups, lam=LAMBDA_MAIN):
    anchor_ids, src, qvecs = query_setup(groups)
    sim = direct_sim(qvecs)
    t_gated = diffuse_local(G, src, rel={n: sim[n] for n in NODE_IDS})
    t_plain = diffuse_local(G, src)
    t_routed = diffuse_local(G, src, c_relatedto=C_RELATEDTO)
    # per-cue overlap (7c control)
    per_cue = []
    for g in groups:
        nid, _ = resolve_group(g)
        if nid is not None:
            per_cue.append(diffuse_local(G, {nid: 100.0}))
    def overlap(n):
        vals = [pc.get(n, 0.0) for pc in per_cue]
        if not vals:
            return 0.0
        prod = 1.0
        for v in vals:
            prod *= (v + 1e-9)
        return prod ** (1.0 / len(vals))
    S = {}
    for n in NODE_IDS:
        m0 = t_plain.get(n, 0.0) / pr_pow(n)               # pure baseline: topology + prior, NO similarity
        mgate = t_gated.get(n, 0.0)                          # legacy control: current pipeline (in-propagation gate)
        m1 = (t_plain.get(n, 0.0) + lam * sim[n]) / pr_pow(n)   # additive-sim bypass (M0 + term)
        m2 = (t_routed.get(n, 0.0) + lam * sim[n]) / pr_pow(n)  # + relation routing
        m3 = m2 / (GEN[n] ** MU)                             # + genericness penalty
        ctrl = overlap(n) / pr_pow(n)                       # 7c overlap control (expect tie)
        S[n] = {"M0": m0, "M_gate": mgate, "M1": m1, "M2": m2, "M3": m3, "CTRL_overlap": ctrl}
    return anchor_ids, set(anchor_ids), sim, S

METHODS = ["M0", "M_gate", "M1", "M2", "M3", "CTRL_overlap"]

## Run + manifold-aware evaluation

Sources excluded from ranking. `strict` = canonical (first manifold node) beats
distractor; `manifold` = any manifold node beats distractor. Rank on
pairwise/solved counts, never on absolute margin (7a lesson).

In [13]:
def best_manifold(sc, man_ids):
    scored = [(t, sc.get(i, 0.0)) for t, i in man_ids]
    return max(scored, key=lambda kv: kv[1]) if scored else (None, 0.0)

def rank_of(order, target_ids):
    pos = {nid: r for r, nid in enumerate(order, start=1)}
    ranks = [pos[i] for i in target_ids if i in pos]
    return min(ranks) if ranks else None

def evaluate(lam=LAMBDA_MAIN):
    agg = {m: {"p_pair": 0, "p_strict": 0, "p_n": 0, "c_pair": 0, "c_n": 0,
               "p_ranks": [], "p_mrr": []} for m in METHODS}
    for kind, qid, groups, manifold, distractors in QUERIES:
        anchors, src_set, sim, S = scores_for_query(groups, lam=lam)
        man = [(t, resolve(t)) for t in manifold]
        man = [(t, i) for t, i in man if i is not None and i not in src_set]
        dist = [resolve(t) for t in distractors]
        dist = [i for i in dist if i is not None and i not in src_set]
        if not man or not dist:
            continue
        for m in METHODS:
            sc = {n: S[n][m] for n in NODE_IDS if n not in src_set}
            dmax = max(sc.get(i, 0.0) for i in dist)   # beat the STRONGEST distractor
            _, best_s = best_manifold(sc, man)
            canonical_s = sc.get(man[0][1], 0.0)
            if kind == "primary":
                agg[m]["p_n"] += 1
                agg[m]["p_pair"] += int(best_s > dmax)
                agg[m]["p_strict"] += int(canonical_s > dmax)
                order = [n for n, _ in sorted(sc.items(), key=lambda kv: -kv[1])]
                r = rank_of(order, [i for _, i in man])
                if r:
                    agg[m]["p_ranks"].append(r)
                    agg[m]["p_mrr"].append(1.0 / r)
            else:
                agg[m]["c_n"] += 1
                agg[m]["c_pair"] += int(best_s > dmax)
    return agg

agg = evaluate(LAMBDA_MAIN)
print(f"lambda = {LAMBDA_MAIN}   (primary: beat MAX distractor; rank = best-manifold rank among non-source nodes; lower is better)\n")
print(f"{'method':14s} | p.manifold | p.strict | mean_rank | MRR   | control")
for m in METHODS:
    a = agg[m]
    mr = (sum(a["p_ranks"]) / len(a["p_ranks"])) if a["p_ranks"] else float("nan")
    mrr = (sum(a["p_mrr"]) / len(a["p_mrr"])) if a["p_mrr"] else float("nan")
    print(f"{m:14s} | {a['p_pair']}/{a['p_n']:<8} | {a['p_strict']}/{a['p_n']:<6} | {mr:8.1f} | {mrr:.3f} | {a['c_pair']}/{a['c_n']}")

lambda = 1.0   (primary: beat MAX distractor; rank = best-manifold rank among non-source nodes; lower is better)

method         | p.manifold | p.strict | mean_rank | MRR   | control
M0             | 2/4        | 1/4      |    232.0 | 0.095 | 2/2
M_gate         | 2/4        | 1/4      |    217.2 | 0.158 | 2/2
M1             | 2/4        | 1/4      |     63.5 | 0.094 | 2/2
M2             | 2/4        | 1/4      |     63.5 | 0.094 | 2/2
M3             | 2/4        | 1/4      |    135.8 | 0.080 | 2/2
CTRL_overlap   | 1/4        | 0/4      |   5021.2 | 0.056 | 2/2


In [14]:
# lambda sweep, reported by MRR (continuous) so good methods do NOT saturate at a shared ceiling
print("primary MRR (higher = target ranked nearer the top); this is what separates M1 from CTRL_overlap")
print(f"{'lambda':>7} | " + " | ".join(f"{m:>12}" for m in METHODS))
for lam in LAMBDAS:
    aa = evaluate(lam)
    def _mrr(m):
        r = aa[m]["p_mrr"]
        return (sum(r) / len(r)) if r else float("nan")
    print(f"{lam:>7} | " + " | ".join(f"{_mrr(m):>12.3f}" for m in METHODS))

primary MRR (higher = target ranked nearer the top); this is what separates M1 from CTRL_overlap
 lambda |           M0 |       M_gate |           M1 |           M2 |           M3 | CTRL_overlap
    0.0 |        0.095 |        0.158 |        0.095 |        0.095 |        0.060 |        0.056
    0.5 |        0.095 |        0.158 |        0.094 |        0.094 |        0.080 |        0.056
    1.0 |        0.095 |        0.158 |        0.094 |        0.094 |        0.080 |        0.056
    2.0 |        0.095 |        0.158 |        0.099 |        0.099 |        0.075 |        0.056
    4.0 |        0.095 |        0.158 |        0.078 |        0.078 |        0.044 |        0.056


## Top-k inspection with provenance (sources excluded)

Show, for a couple of methods, what actually surfaces + per-anchor support, so we
can see whether the target manifold appears and whether the answer is a
multi-cue intersection or a one-cue hub.

In [15]:
def topk(groups, method, k=12, lam=LAMBDA_MAIN):
    anchors, src_set, sim, S = scores_for_query(groups, lam=lam)
    sc = [(n, S[n][method]) for n in NODE_IDS if n not in src_set]
    sc.sort(key=lambda kv: -kv[1])
    print(f"  anchors: {[G.nodes[a].content for a in anchors]}")
    for n, s in sc[:k]:
        print(f"    {s:9.3f}  sim={sim[n]:.2f} pr%={pr_pct[n]:.2f} gen={GEN[n]:.2f}  {G.nodes[n].content!r}")

for kind, qid, groups, manifold, distractors in QUERIES:
    if kind != "primary":
        continue
    print(f"\n===== {qid}  target~{manifold} vs {distractors} =====")
    for m in ("M0", "M1", "CTRL_overlap"):   # the key unresolved comparison
        print(f"  --- {m} ---")
        topk(groups, m)


===== Q1  target~['caffeine', 'stimulant'] vs ['beverage', 'hot beverage', 'popular drink'] =====
  --- M0 ---
  anchors: ['coffee', 'stimulant']
       56.228  sim=1.00 pr%=0.88 gen=1.88  'served hot'
       40.130  sim=0.00 pr%=0.77 gen=1.77  'made from coffee beans'
       35.248  sim=0.73 pr%=0.95 gen=1.95  'coffee shop'
       34.190  sim=0.64 pr%=0.99 gen=1.99  'caffeine'
       29.493  sim=0.00 pr%=0.68 gen=1.68  'good in morning'
       29.493  sim=0.63 pr%=0.68 gen=1.68  'addictive substance'
       29.493  sim=0.00 pr%=0.68 gen=1.68  'bushy plant'
       29.493  sim=0.00 pr%=0.68 gen=1.68  'good after dinner drink'
       29.493  sim=0.00 pr%=0.68 gen=1.68  'popular drink'
       23.421  sim=0.42 pr%=0.92 gen=1.92  'acquired taste'
       20.994  sim=0.83 pr%=0.94 gen=1.94  'caffein'
       15.844  sim=0.00 pr%=0.92 gen=1.92  'shop (n)'
  --- M1 ---
  anchors: ['coffee', 'stimulant']
       66.576  sim=1.00 pr%=0.88 gen=1.88  'served hot'
       41.372  sim=0.73 pr%=0.95 gen

## Interpretation checklist

1. Does `M1` (additive sim) move **primary manifold-pairwise** above `M0` and above `CTRL_overlap`?
2. Does `M2` (relation routing) add anything on top of `M1`?
3. Does `M3` (genericness) help without breaking controls?
4. Is the movement a real ordering change, or just magnitude (check top-k, not margins)?
5. `strict` vs `manifold` gap: how often does the graph answer the manifold but not the canonical node? (feeds exp9)
6. Does `CTRL_overlap` tie, as predicted? If it moves, that itself is news.
7. Lambda sweep: is there a stable region where sim helps before it collapses toward pure vector-RAG (M1 ~ direct-similarity ranking)?